# Evaluate PPO against fixed timers in SUMO
Paired offline comparison using identical generated demand for each policy, scenario and seed. Use validation seeds for model selection and reserve a different set for the final report. No live control.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'traffic_training').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the Train folder or its notebooks subfolder.')
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image


In [ ]:
MODEL = ROOT / 'outputs' / 'rl' / 'policy.zip'
OUTPUT = ROOT / 'outputs' / 'rl-test'
SEEDS = [2000001, 2000002, 2000003]
SCENARIOS = ['balanced', 'ns_peak', 'ew_peak', 'burst']
FIXED_SECONDS = [30, 60]
HORIZON_SECONDS = 1800
GUI = False
DEVICE = 'cpu'


In [ ]:
import json
from traffic_training.sumo_env import SumoConfig
from traffic_training.rl import evaluate_policy
settings = json.loads(MODEL.with_name('policy_metadata.json').read_text())['sumo_config']
settings.update(network_dir=str(ROOT / 'sumo' / 'junction'), horizon_seconds=HORIZON_SECONDS, gui=GUI, sumo_binary=None)
settings['durations'] = tuple(settings['durations'])
config = SumoConfig(**settings)
episodes, summary = evaluate_policy(MODEL, OUTPUT, config=config, seeds=SEEDS,
    scenarios=SCENARIOS, fixed_seconds=FIXED_SECONDS, device=DEVICE)
display(summary)


In [ ]:
import pandas as pd
display(Image(filename=str(OUTPUT / 'queue_comparison.png')))
display(pd.read_csv(OUTPUT / 'paired_queue_deltas.csv'))
display(pd.read_csv(OUTPUT / 'phase_traces.csv').head(12))


Lower queue and negative PPO-minus-fixed queue deltas favor PPO. Compare scenarios, variance, throughput, unfinished demand and collisions rather than reporting one favorable seed. Final phases can be partially observed at the shared time horizon; they are marked incomplete without triggering an extra transition. This simple junction and ground-truth SUMO observations do not establish real-world performance. The standalone equivalent is `scripts/evaluate_rl.py`.